# Mục 3b(i) — Điền giá trị thiếu
**Võ Minh Huy — 2311110117**

Thực hiện N01–N07 đã mô tả trong Word. Giữ nguyên CSV nguồn; không dùng original để tra đáp án. Kết quả là dữ liệu trung gian, chưa phải after cuối. Đọc code ở file `04_dien_null.py` để xem toàn bộ thuật toán.

In [1]:
from pathlib import Path
import importlib.util

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
spec = importlib.util.spec_from_file_location(
    "dien_null", thu_muc / "python" / "04_dien_null.py"
)
dien_null = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dien_null)
ket_qua = dien_null.run()
ket_qua["chuan_bi_python.csv"]

source_rows,empty_rows_removed,duplicate_rows_removed,working_rows,cells_to_impute,filled_cells,unresolved_cells,invalid_target_rows
309,5,5,299,126,126,0,1


## 1. Kết quả điền theo từng quy tắc
Mẫu cho được cố định trước điền. Nhóm dưới 5 mẫu chuyển sang cấp dự phòng. Chỉ sửa các ô thiếu hoặc sai đã gắn cờ trên bản làm việc.

In [2]:
ket_qua["tong_hop_quy_tac_python.csv"]

rule_id,column_name,missing_cells,filled_cells,fallback_cells,min_donors,max_donors
N01,sex,20,20,0,8,32
N02,age,23,23,0,45,103
N03,creatinine_phosphokinase,21,21,0,24,41
N04,platelets,20,20,0,43,104
N05,serum_creatinine,20,20,0,12,58
N06,serum_sodium,21,21,0,19,48
N07,ejection_fraction,1,1,0,56,56


## 2. Minh họa nhật ký
Giữ mã hồ sơ, dòng CSV, nguyên nhân thiếu, nhóm sử dụng, số mẫu cho và giá trị điền.

In [3]:
nhat_ky = ket_qua["nhat_ky_dien_python.csv"]
cot_hien_thi = ["rule_id", "source_csv_line", "column_name", "old_value",
                "missing_reason", "priority", "donor_count", "imputed_value"]
nhat_ky.groupby("rule_id", sort=False).head(2)[cot_hien_thi]

rule_id,source_csv_line,column_name,old_value,missing_reason,priority,donor_count,imputed_value
N01,11,sex,NaN,MISSING,1,9,1.000000
N01,17,sex,NaN,MISSING,1,29,1.000000
N02,7,age,NaN,MISSING,1,103,60.000000
N02,11,age,NaN,MISSING,1,103,60.000000
N03,4,creatinine_phosphokinase,NaN,MISSING,1,40,267.000000
N03,17,creatinine_phosphokinase,99999,INVALID_VALUE,1,24,160.000000
N04,32,platelets,NaN,MISSING,1,104,259763.430000
N04,38,platelets,NaN,MISSING,1,65,247970.370000
N05,17,serum_creatinine,NaN,MISSING,1,33,1.200000
N05,46,serum_creatinine,NaN,MISSING,1,58,1.250000


## 3. Trường hợp dùng nhóm dự phòng

In [4]:
nhat_ky.loc[nhat_ky["priority"] > 1, ["rule_id", "source_csv_line", "group_description", "donor_count", "imputed_value"]]

rule_id,source_csv_line,group_description,donor_count,imputed_value


## 4. Kiểm chứng bằng cách che 20% dữ liệu đã biết
Che độc lập từng cột. Thứ tự che dùng hash với seed 42, giống SQL. Mẫu bị che được loại khỏi tập mẫu cho và khóa nhóm dẫn xuất. Accuracy: cao hơn tốt hơn; MAE: thấp hơn tốt hơn. So với phương án nền toàn cột, không mặc định phương pháp nhóm sẽ tốt hơn.

In [5]:
ket_qua["kiem_chung_che_20_phan_tram_python.csv"]

rule_id,column_name,hidden_count,predicted_count,metric,group_score,baseline_score,baseline_value,better_than_baseline
N01,sex,56,56,accuracy,0.5892857143,0.7500000000,1.000000,False
N02,age,56,56,MAE,9.0047678571,9.0297678571,60.000000,True
N03,creatinine_phosphokinase,56,56,MAE,568.3035714286,567.7678571429,246.000000,False
N04,platelets,56,56,MAE,77242.2675000000,77937.3239285714,258664.510000,True
N05,serum_creatinine,56,56,MAE,0.4641071429,0.4605357143,1.100000,False
N06,serum_sodium,56,56,MAE,3.1964285714,2.8035714286,136.000000,False
N07,ejection_fraction,60,60,MAE,9.8500000000,9.7833333333,38.000000,False


## 5. Nhãn kết quả không được tự điền
Hồ sơ có DEATH_EVENT = 2 vẫn được giữ, gắn cờ; không đưa vào phân tích cần nhãn hợp lệ.

In [6]:
du_lieu = ket_qua["du_lieu_sau_dien_python.csv"]
du_lieu.loc[du_lieu["invalid_target"], ["source_csv_line", "DEATH_EVENT", "invalid_target"]]

source_csv_line,DEATH_EVENT,invalid_target
62,2.000000,True


Code đầy đủ, dữ liệu trung gian, nhật ký và kết quả kiểm chứng được lưu trong thư mục `ket_qua/04_dien_null`. Các kết quả này phải được đối chiếu từng ô với SQL trước khi kết luận hoàn thành mục 3b(i).